# 01 — EDA e Qualidade

Objetivos:
- Inspecionar a base, tipos e ausentes
- Entender duplicidades (CLIENTE aparece em múltiplas linhas) e validar causa
- Diagnosticar outliers em engajamento/inatividade
- Gerar um data quality report para apoiar as decisões de limpeza e modelagem

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))


import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from src.preprocessing import (
    clean_base,
    data_quality_report,
    diagnose_repetition_cause,
    load_xlsx_base_clientes,
    winsorize_df,
)
from src.viz import corr_heatmap, hist_kde, plot_missingness, set_style

set_style()

PROJECT_ROOT = Path('..').resolve()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Base Customer Sucess.xlsx'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_raw = load_xlsx_base_clientes(str(RAW_PATH))
df_raw.shape

In [ ]:
df_raw.head(3)

## 1) Qualidade: tipos, ausentes, cardinalidade

In [ ]:
df, meta = clean_base(df_raw)
df.shape, meta

In [ ]:
rep = data_quality_report(df)
rep.head(10)

In [ ]:
rep.to_excel(REPORTS_DIR / 'data_quality_report.xlsx', index=False)
ax = plot_missingness(rep, top_n=20)
plt.show()

## 2) Duplicidades e unidade correta (cliente vs. transação)

A base está em granularidade transacional (uma linha por transação/ordem). Para segmentação de **clientes**, precisamos consolidar para 1 linha por CLIENTE (visão cliente-atual) e, ao mesmo tempo, capturar o histórico via agregações (ex.: número de transações e recência).

Nesta seção, validamos a causa da repetição de CLIENTE.

In [ ]:
dup_cliente = df['cliente'].duplicated().mean()
dup_transacao = df['transacao'].duplicated().mean() if 'transacao' in df.columns else np.nan
pd.Series({'pct_linhas_com_cliente_repetido': dup_cliente, 'pct_transacao_repetida': dup_transacao})

In [ ]:
rep_causa = diagnose_repetition_cause(df)
rep_causa.head(10)

In [ ]:
rep_causa['tem_multiplas_transacoes'].mean(), rep_causa['n_transacoes'].describe()

In [ ]:
top_clientes = rep_causa.sort_values(['n_linhas','n_transacoes'], ascending=False).head(5)['cliente'].tolist()
cols_debug = [c for c in ['cliente','transacao','data_ordem','total_parcelas','recorrente','status'] if c in df.columns]
df[df['cliente'].isin(top_clientes)][cols_debug].sort_values(['cliente','data_ordem']).head(50)

Conclusão esperada ao final do notebook (a ser validada pelos outputs acima):
- Se a maioria dos clientes repetidos tiver `n_transacoes > 1`, a repetição é causada por múltiplas compras/renovações (granularidade transacional).
- Para o case de segmentação de clientes, a unidade correta é **cliente**, consolidando 1 linha por cliente (registro mais recente) e preservando histórico via features agregadas.

## 3) Outliers: n_acessos e dias_sem_acessar

In [ ]:
for col in ['n_acessos','dias_sem_acessar']:
    if col in df.columns:
        q = df[col].quantile([0.5,0.75,0.9,0.95,0.99]).to_frame('quantil')
        display(col, q)

In [ ]:
df_w = winsorize_df(df, cols=[c for c in ['n_acessos','dias_sem_acessar'] if c in df.columns], lower_q=0.01, upper_q=0.99)

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
sns.boxplot(x=df['n_acessos'], ax=ax[0,0]); ax[0,0].set_title('n_acessos (original)')
sns.boxplot(x=df_w['n_acessos'], ax=ax[0,1]); ax[0,1].set_title('n_acessos (winsor 1-99%)')
sns.boxplot(x=df['dias_sem_acessar'], ax=ax[1,0]); ax[1,0].set_title('dias_sem_acessar (original)')
sns.boxplot(x=df_w['dias_sem_acessar'], ax=ax[1,1]); ax[1,1].set_title('dias_sem_acessar (winsor 1-99%)')
plt.tight_layout(); plt.show()

Decisão de tratamento (usada nos próximos notebooks):
- `n_acessos`: usar `log1p` (já criado como `log_acessos`) para reduzir assimetria e sensibilidade a outliers.
- `dias_sem_acessar`: aplicar winsorização (1%–99%) apenas se necessário (verificado via boxplot/quantis).

## 4) Visualizações rápidas (numéricas/categóricas) + correlação

In [ ]:
for col in ['n_acessos','log_acessos','dias_sem_acessar','recorrencia']:
    if col in df.columns:
        plt.figure();
        hist_kde(df, col)
        plt.show()

In [ ]:
if 'ativo' in df.columns:
    ax = sns.countplot(data=df, y='ativo', order=df['ativo'].value_counts().index)
    ax.set_title('Distribuição: ativo')
    plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr_heatmap(df)
plt.show()